# Water Rights Restored to the Gila River

The impacts of irrigation on vegetation health in the Gila River Valley

## Set up: import libraries

We’ll need some Python libraries to complete this workflow.

In the cell below, making sure to keep the packages in order, add packages for:

- Working with DataFrames  
- Working with GeoDataFrames  
- Making interactive plots of tabular and vector data


### Reflect and Respond #1: 
What are we using the rest of these packages for? See if you can figure it out as you complete the notebook.

In [2]:
### working with dataframes
import pandas as pd

### working with geodataframes
import geopandas as gpd

### making interactive plots of tabular and vector data
import hvplot.pandas  # DataFrame and GeoDataFrame plots
import hvplot.xarray  # Raster plots
import holoviews as hv
import panel as pn

pn.extension(comms='vscode', inline=True)
hv.extension('bokeh')

### work with json data
import json

### work with pattern matching rules for pathnames 
from glob import glob

### access the data we'll use for the assignment
import earthpy

### interactive plots with rasters 
import hvplot.xarray

### handle geospatial rasters
import rioxarray as rxr

### work with arrays
import xarray as xr

print("Yay! Imports!")

Yay! Imports!


## Step 1: Download sample data

In this analysis, you’ll need to download multiple data files to your computer. You’ll need to set up a folder/directory for the files, and while you’re at it, download the sample data there.

In the cell below, replace ‘Project Name’ with ‘Gila River Vegetation' and 'my-data-folder' with a **descriptive** directory name.

In [3]:
### define the project and data
project = earthpy.Project(
    'Gila River Vegetation', dirname = 'gila-river-data-folder')

### get the data
project.get_data()

### Earth Data Science data formats:

| Data type | Descriptions | Common file formats | Python type |
|------------------|------------------|------------------|------------------|
| Time Series | The same data points (e.g. streamflow) collected multiple times over time | Tabular formats (e.g. .csv, or .xlsx) | pandas DataFrame |
| Vector | Points, lines, and polygons (with coordinates) | Shapefile (often an archive like a `.zip` file because a Shapefile is actually a collection of at least 3 files) | geopandas GeoDataFrame |
| Raster | Evenly spaced spatial grid (with coordinates) | GeoTIFF (`.tif`), NetCDF (`.nc`), HDF (`.hdf`) | rioxarray DataArray |

Read more about [vector](https://www.earthdatascience.org/courses/use-data-open-source-python/intro-vector-data-python/spatial-data-vector-shapefiles/) and [raster](https://earthdatascience.org/courses/intro-to-earth-data-science/file-formats/use-spatial-data/use-raster-data/) data in the Earth Data Science textbook.

### Reflect and Respond #2:
For this coding challenge, we are interested in the boundary of the Gila River Indian Community, and the health of vegetation in the area measured on a scale from -1 to 1. In the cell below, answer the following questions: What data type do you think the boundary will be? What about the vegetation health?

## Step 2: Make a site map

### Step 2A: Load the Gila River Indian Community boundary  

- Locate the Tribal Subdivision files in your download directory  
- Change `'subdivision-directory'` to the actual location
- Load the data into Python and check that it worked


In [4]:
### figure out where the download lives
print(project.project_dir)

C:\Users\gekek\AppData\Local\earth-analytics\earth-analytics\gila-river-data-folder


In [5]:
aitsn_gdf = gpd.read_file(
    project.project_dir/'tl_2020_us_aitsn'/'tl_2020_us_aitsn.shp'
)

You might notice in this dataset that some of the names are not easily searchable. For example, the Gila River subdivisions are named “District 1-7”! So, how do we know what to search for? We recommend making an **interactive** plot of the data so that you can find the information you need:

In [6]:
# Convert to Web Mercator to match the satellite basemap (KeyError: 'lon_0' is a geographic plotting compatibility issue)
### make an interactive map
aitsn_map = aitsn_gdf.to_crs(epsg=3857).hvplot(

    ### set basemap
    geo = False, tiles = 'EsriImagery', 

    ### set figure size
    frame_width = 500,

    ### other map parameters
    legend = False, fill_color = None, line_color = 'white',

    ### make all the column values in the dataset visible
    hover_cols = 'all')

# Display the map
pn.panel(aitsn_map)

BokehModel(combine_events=True, render_bundle={'docs_json': {'0700a539-0e92-4ba3-a7cc-19b1700c26e7': {'version…

### Reflect and Respond #3:
What column could you use to uniquely identify the subdivisions of the reservation you want to study using this interactive map? What value do you need to use to filter the `GeoDataFrame`?

Now that you have the info you need, it’s also a good idea to check the data type. Are the values in the unique identifier column you've selected some kind of **number** or an **object** like a text string? We can’t tell just by looking, which is where the `.info()` method comes in:

In [7]:
aitsn_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 484 entries, 0 to 483
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype   
---  ------    --------------  -----   
 0   AIANNHCE  484 non-null    str     
 1   TRSUBCE   484 non-null    str     
 2   TRSUBNS   484 non-null    str     
 3   GEOID     484 non-null    str     
 4   NAME      484 non-null    str     
 5   NAMELSAD  484 non-null    str     
 6   LSAD      484 non-null    str     
 7   CLASSFP   484 non-null    str     
 8   MTFCC     484 non-null    str     
 9   FUNCSTAT  484 non-null    str     
 10  ALAND     484 non-null    int64   
 11  AWATER    484 non-null    int64   
 12  INTPTLAT  484 non-null    str     
 13  INTPTLON  484 non-null    str     
 14  geometry  484 non-null    geometry
dtypes: geometry(1), int64(2), str(12)
memory usage: 93.7 KB


### Reflect and Respond #4:
What is the data type of the unique identifier column? How will that affect your code?

## Step 2B: Select the Gila River subdivisions and make a site map.

- Replace `identifier` with the value you found from exploring the interactive map. Make sure that you are using the correct data type!  
- Change the plot to have a web tile basemap and add a title. Change the color of the Gila River Indian Community polygon so that it's transparent

In [8]:
### select and merge the Gila River subdivisions
gila_river_gdf = aitsn_gdf.loc[
    aitsn_gdf.AIANNHCE == '1310'
].dissolve()

### plot a transparent boundary over satellite imagery
gila_river_map = gila_river_gdf.to_crs(epsg=3857).hvplot(
    geo=False,
    tiles='EsriImagery',
    title='Gila River Indian Community',
    alpha=0.4,
    frame_width=500,
    legend=False,
    fill_color=None,
    line_color='white',
    hover_cols='all'
)

# Display the map
pn.panel(gila_river_map)

BokehModel(combine_events=True, render_bundle={'docs_json': {'700c4ff7-388f-4a05-b1a8-51ffcaffc5cd': {'version…

## Wrap up Steps 1-2

Don’t forget to store your variables so you can use them in other notebooks! Replace `var1` and `var2` with the variable you want to save, separated by spaces.

In [9]:
%store project gila_river_gdf

Stored 'project' (Project)
Stored 'gila_river_gdf' (GeoDataFrame)


Finally, be sure to `Restart` and `Run all` to make sure your notebook works all the way through!